# SageMaker Training & Deployment Walkthrough

A step-by-step guide to training and deploying a **loan approval prediction** model on Amazon SageMaker.

### What You'll Learn

1. Preprocess a loan approval dataset (feature engineering)
2. Train a `RandomForestClassifier` using `ModelTrainer`
3. Deploy the model to a real-time endpoint using `ModelBuilder`
4. Test with synchronous real-time invocation and asynchronous Batch Transform
5. Clean up resources

### Model Architecture

```
Raw CSV → [Preprocessing: categorical remapping, log transforms, feature selection]
        → [Training: ColumnTransformer (MinMaxScaler + OneHotEncoder) → RandomForestClassifier]
        → [Inference: predict_proba → positive-class probability]
```

### Prerequisites

- `sagemaker` SDK installed (V3)
- `ipywidgets` for deployment progress display
- AWS credentials configured with SageMaker access

---
## Step 1: Setup and Configuration

Initialize the SageMaker session and define the feature/hyperparameter settings for our loan approval model.

In [ ]:
import uuid
import os
import platform
import tempfile

import numpy as np
import pandas as pd
import boto3

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, InputData
from sagemaker.serve import ModelBuilder
from sagemaker.core.transformer import Transformer

sagemaker_session = Session()
if sagemaker_session.default_bucket_prefix:
    sagemaker_session.default_bucket_prefix = (
        sagemaker_session.default_bucket_prefix.strip("/")
    )

region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
unique_id = str(uuid.uuid4())[:8]
prefix = f"notebook-walkthrough/{unique_id}"

# Feature and model configuration
LABEL_COLUMN = "approved"
CATEGORICAL_FEATURES = ["dep_cat"]
NUMERICAL_FEATURES = ["loan_term", "cibil_score", "log_residential_assets_value", "log_loan_amount"]
LOG_TRANSFORM_COLUMNS = ["residential_assets_value", "loan_amount"]
SELECTED_FEATURES = ["dep_cat", "loan_term", "cibil_score", "log_residential_assets_value", "log_loan_amount"]
PREDICTION_THRESHOLD = 0.6

HYPERPARAMETERS = {
    "n_estimators": 200,
    "max_depth": 5,
    "class_weight": "balanced_subsample",
    "n_jobs": 2,
    "random_state": 42,
}

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Prefix: {prefix}")

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Region: ap-southeast-1
Bucket: amazon-sagemaker-187833180105-ap-southeast-1-bc9lwfye753b1d
Prefix: notebook-walkthrough/ffb65a97
Role: arn:aws:iam::187833180105:role/SageMakerRole


---
## Step 2: Data Preparation

Load the raw loan approval dataset and apply feature engineering:

1. Remap `no_of_dependents` → categorical `dep_cat` (none/one/couple/many)
2. Log-transform skewed monetary columns (`residential_assets_value`, `loan_amount`)
3. Select 5 features for the model
4. Train/test split (80/20)

In [2]:
from sklearn.model_selection import train_test_split

DEPENDENTS_MAP = {0: "none", 1: "one", 2: "couple"}

raw_df = pd.read_csv("../data/loan_approval_dataset.csv")
print(f"Raw dataset: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

# Drop identifier
df = raw_df.drop(columns=["loan_id"], errors="ignore")

# Remap no_of_dependents → dep_cat
df["dep_cat"] = df["no_of_dependents"].apply(lambda x: DEPENDENTS_MAP.get(int(x), "many"))

# Log-transform skewed monetary columns
for col in LOG_TRANSFORM_COLUMNS:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

# Select features + label
df = df[SELECTED_FEATURES + [LABEL_COLUMN]]

# Train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} rows")
print(f"Test: {len(test_df)} rows")
train_df.head()

Raw dataset: 4269 rows, 13 columns
Train: 3415 rows
Test: 854 rows


,dep_cat,loan_term,cibil_score,log_residential_assets_value,log_loan_amount,approved
1675,many,6,568,15.573369,17.213369,1
1164,none,12,710,16.985196,17.341871,1
192,one,8,682,14.603968,14.880222,1
910,couple,18,754,15.919645,16.388123,1
567,many,12,441,15.955577,16.222456,0


In [3]:
# Upload training data to S3
data_dir = tempfile.mkdtemp()
train_file = os.path.join(data_dir, "train.csv")
train_df.to_csv(train_file, index=False)

s3_client = boto3.client("s3")
train_s3_key = f"{prefix}/data/train.csv"
s3_client.upload_file(train_file, bucket, train_s3_key)
train_data_uri = f"s3://{bucket}/{train_s3_key}"
print(f"Training data uploaded: {train_data_uri}")

Training data uploaded: s3://amazon-sagemaker-187833180105-ap-southeast-1-bc9lwfye753b1d/notebook-walkthrough/ffb65a97/data/train.csv


---
## Step 3: Train the Model with ModelTrainer

`ModelTrainer` is SageMaker's high-level training API. The training script (`src/train.py`) builds a sklearn `Pipeline`:
- **ColumnTransformer**: `MinMaxScaler` for numerical features + `OneHotEncoder` for categorical
- **Classifier**: `RandomForestClassifier`

The trained pipeline is saved as `model.joblib` in the model artifacts.

In [4]:
source_dir = os.path.join(os.getcwd(), "src")

sklearn_image_uri = image_uris.retrieve("sklearn", region, version="1.4-2")
print(f"SKLearn image: {sklearn_image_uri}")
print(f"Source directory: {source_dir}")
print(f"Contents: {os.listdir(source_dir)}")

SKLearn image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3
Source directory: /mnt/custom-file-systems/s3/shared/notebooks/src
Contents: ['inference.py', 'train.py']


In [ ]:
model_trainer = ModelTrainer(
    training_image=sklearn_image_uri,
    source_code=SourceCode(
        source_dir=source_dir,
        entry_script="train.py",
    ),
    hyperparameters={
        "n-estimators": str(HYPERPARAMETERS["n_estimators"]),
        "max-depth": str(HYPERPARAMETERS["max_depth"]),
        "class-weight": str(HYPERPARAMETERS["class_weight"]),
        "n-jobs": str(HYPERPARAMETERS["n_jobs"]),
        "random-state": str(HYPERPARAMETERS["random_state"]),
        "label-column": LABEL_COLUMN,
        "categorical-features": ",".join(CATEGORICAL_FEATURES),
        "numerical-features": ",".join(NUMERICAL_FEATURES),
    },
    base_job_name="notebook-loan-train",
    sagemaker_session=sagemaker_session,
)

model_trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=train_data_uri)]
)

model_artifacts_uri = model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
print(f"Model artifacts: {model_artifacts_uri}")

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Output()

Model artifacts: s3://amazon-sagemaker-187833180105-ap-southeast-1-bc9lwfye753b1d/shared/notebook-loan-train/notebook-loan-train-20261001092503/output/model.tar.gz


---
## Step 4: Deploy with ModelBuilder

`ModelBuilder` is the recommended way to deploy models in SageMaker V3. It handles model packaging and endpoint creation in a single flow.

The inference script (`src/inference.py`) defines the four handler functions the sklearn container expects:
- `model_fn` — loads the trained model
- `input_fn` — parses headerless CSV into a DataFrame
- `predict_fn` — returns positive-class probability
- `output_fn` — serializes predictions as CSV

In [ ]:
# ModelBuilder needs a local temp path for model artifacts
model_path = os.path.join(tempfile.gettempdir(), "sagemaker", "model-builder", unique_id)
os.makedirs(model_path, exist_ok=True)

model_builder = ModelBuilder(
    image_uri=sklearn_image_uri,
    s3_model_data_url=model_artifacts_uri,
    model_path=model_path,
    sagemaker_session=sagemaker_session,
    dependencies={},
    source_code=SourceCode(
        source_dir=source_dir,
        entry_script="inference.py",
        requirements=None,
    ),
)

model = model_builder.build()
print(f"Model built: {model.model_name}")

endpoint_name = f"notebook-loan-{unique_id}"

endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    instance_type="ml.m5.large",
    initial_instance_count=1,
)

print(f"Endpoint deployed: {endpoint_name}")

Model built: model-9b5fc578


Output()

Endpoint deployed: notebook-loan-ffb65a97


---
## Step 5: Testing (Synchronous API Invocation)

Retrieve the deployed endpoint by name and send the full test set for real-time inference. Using `Endpoint.get()` decouples this step from the deployment object — only the endpoint name is needed, which is how a separate inference service would consume it.

- **FDR** (False Discovery Rate) and **Recall** — primary gating metrics
- **Accuracy**, **Precision**, **F1** — informational metrics
- **AUC** — threshold-independent discrimination

Predictions are binarized at the configured `PREDICTION_THRESHOLD`.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sagemaker.core.resources import Endpoint

# Column order: numerical features first, then categorical
column_order = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

test_samples = test_df[column_order]
test_labels = test_df[LABEL_COLUMN].values

# Format as headerless CSV
csv_data = "\n".join(
    ",".join(str(v) for v in row) for row in test_samples.values
)

# Endpoint.get expects a boto3.session.Session, not sagemaker.core.helper.session_helper.Session
rt_endpoint = Endpoint.get(endpoint_name, session=sagemaker_session.boto_session)
response = rt_endpoint.invoke(
    body=csv_data,
    content_type="text/csv",
    accept="text/csv",
)

result = response.body.read().decode("utf-8")
probabilities = np.array([float(x) for x in result.strip().split(",")])
predictions = (probabilities >= PREDICTION_THRESHOLD).astype(int)

cm = confusion_matrix(test_labels, predictions)
tn, fp, fn, tp = cm.ravel()
fdr = float(fp / (tp + fp)) if (tp + fp) > 0 else 0.0

print(f"Prediction threshold: {PREDICTION_THRESHOLD}")
print(f"Test samples: {len(test_labels)}")
print(f"\n{'Metric':<12} {'Value'}")
print("-" * 25)
print(f"{'FDR':<12} {fdr:.4f}")
print(f"{'Recall':<12} {recall_score(test_labels, predictions):.4f}")
print(f"{'Accuracy':<12} {accuracy_score(test_labels, predictions):.4f}")
print(f"{'Precision':<12} {precision_score(test_labels, predictions, average='weighted', zero_division=0):.4f}")
print(f"{'F1':<12} {f1_score(test_labels, predictions, average='weighted', zero_division=0):.4f}")
print(f"{'AUC':<12} {roc_auc_score(test_labels, probabilities):.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")

Prediction threshold: 0.6
Test samples: 854

Metric       Value
-------------------------
FDR          0.0020
Recall       0.9347
Accuracy     0.9578
Precision    0.9617
F1           0.9582
AUC          0.9957

Confusion Matrix:
  TN=317  FP=1
  FN=35  TP=501


---
## Step 6: Testing (Asynchronous Batch Inference)

Use **SageMaker Batch Transform** for offline inference over data stored in S3. Unlike a real-time endpoint invocation, the transform job runs asynchronously and does not require an active endpoint.

The uploaded CSV places the ground-truth label in the first column. `input_filter="$[1:]"` removes it before each record reaches the inference container, while the labels remain available locally for evaluation.

In [8]:
# Create a label-first, headerless batch input file and upload it to S3
batch_input = test_df[[LABEL_COLUMN] + column_order]
batch_input_file = os.path.join(data_dir, "batch_test.csv")
batch_input.to_csv(batch_input_file, index=False, header=False)

batch_input_key = f"{prefix}/batch-input/batch_test.csv"
s3_client.upload_file(batch_input_file, bucket, batch_input_key)
batch_input_uri = f"s3://{bucket}/{batch_input_key}"
batch_output_uri = f"s3://{bucket}/{prefix}/batch-output"

print(f"Batch input: {batch_input_uri}")
print(f"Batch output: {batch_output_uri}")

Batch input: s3://amazon-sagemaker-187833180105-ap-southeast-1-bc9lwfye753b1d/notebook-walkthrough/ffb65a97/batch-input/batch_test.csv
Batch output: s3://amazon-sagemaker-187833180105-ap-southeast-1-bc9lwfye753b1d/notebook-walkthrough/ffb65a97/batch-output


In [9]:
transformer = Transformer(
    model_name=model.model_name,
    instance_count=1,
    instance_type="ml.m5.large",
    accept="text/csv",
    assemble_with="Line",
    output_path=batch_output_uri,
    sagemaker_session=sagemaker_session,
)

# Start the managed job without blocking this notebook cell
transformer.transform(
    data=batch_input_uri,
    content_type="text/csv",
    split_type="Line",
    input_filter="$[1:]",
    wait=False,
)

batch_job_name = transformer.latest_transform_job.transform_job_name
print(f"Started Batch Transform job: {batch_job_name}")
print("The job is now running asynchronously in SageMaker.")

Started Batch Transform job: model-9b5fc578-2026-10-01-09-34-52-172
The job is now running asynchronously in SageMaker.


In [10]:
# Wait only when the results are needed, then download the generated .out file
transformer.latest_transform_job.wait(logs=True)

batch_output_key = f"{prefix}/batch-output/batch_test.csv.out"
batch_output_file = os.path.join(data_dir, "batch_test.csv.out")
s3_client.download_file(bucket, batch_output_key, batch_output_file)

with open(batch_output_file, encoding="utf-8") as f:
    batch_result = f.read().strip()

# A transform job may assemble mini-batch responses on separate lines
batch_probabilities = np.fromstring(batch_result.replace("\n", ","), sep=",")
if len(batch_probabilities) != len(test_labels):
    raise ValueError(
        f"Expected {len(test_labels)} predictions, received {len(batch_probabilities)}"
    )

batch_predictions = (batch_probabilities >= PREDICTION_THRESHOLD).astype(int)
batch_cm = confusion_matrix(test_labels, batch_predictions)
batch_tn, batch_fp, batch_fn, batch_tp = batch_cm.ravel()
batch_fdr = (
    float(batch_fp / (batch_tp + batch_fp)) if (batch_tp + batch_fp) > 0 else 0.0
)

print(f"Batch Transform job: {batch_job_name}")
print(f"Prediction threshold: {PREDICTION_THRESHOLD}")
print(f"Test samples: {len(test_labels)}")
print(f"\n{'Metric':<12} {'Value'}")
print("-" * 25)
print(f"{'FDR':<12} {batch_fdr:.4f}")
print(f"{'Recall':<12} {recall_score(test_labels, batch_predictions):.4f}")
print(f"{'Accuracy':<12} {accuracy_score(test_labels, batch_predictions):.4f}")
print(f"{'Precision':<12} {precision_score(test_labels, batch_predictions, average='weighted', zero_division=0):.4f}")
print(f"{'F1':<12} {f1_score(test_labels, batch_predictions, average='weighted', zero_division=0):.4f}")
print(f"{'AUC':<12} {roc_auc_score(test_labels, batch_probabilities):.4f}")
print("\nConfusion Matrix:")
print(f"  TN={batch_tn}  FP={batch_fp}")
print(f"  FN={batch_fn}  TP={batch_tp}")

Output()

Batch Transform job: model-9b5fc578-2026-10-01-09-34-52-172
Prediction threshold: 0.6
Test samples: 854

Metric       Value
-------------------------
FDR          0.0020
Recall       0.9347
Accuracy     0.9578
Precision    0.9617
F1           0.9582
AUC          0.9957

Confusion Matrix:
  TN=317  FP=1
  FN=35  TP=501


---
## Step 7: Clean Up

Delete the endpoint and model to avoid ongoing charges.

In [11]:
try:
    endpoint.delete()
    print(f"Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"Error deleting endpoint: {e}")

try:
    sagemaker_session.sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"Deleted endpoint config: {endpoint_name}")
except Exception as e:
    print(f"Error deleting endpoint config: {e}")

try:
    model.delete()
    print(f"Deleted model: {model.model_name}")
except Exception as e:
    print(f"Error deleting model: {e}")

print("\nCleanup complete!")

Deleted endpoint: notebook-loan-ffb65a97
Deleted endpoint config: notebook-loan-ffb65a97
Deleted model: model-9b5fc578

Cleanup complete!


---
## Summary

| Step | API | What it does |
|------|-----|-------------|
| Preprocessing | Local pandas/numpy | Feature engineering: remapping, log transforms, selection |
| Training | `ModelTrainer` | Runs `train.py` on a managed SageMaker instance |
| Deployment | `ModelBuilder` | Packages model + inference code, creates endpoint |
| Synchronous testing | `endpoint.invoke()` | Sends a CSV payload to an active real-time endpoint |
| Batch testing | `Transformer.transform()` | Runs an asynchronous offline inference job over S3 data |
| Cleanup | `endpoint.delete()` | Tears down resources to stop charges |

### Key Concepts

- **ModelTrainer** handles training job orchestration (instance provisioning, data download, artifact upload)
- **ModelBuilder** handles model packaging (repacking artifacts with inference code) and endpoint lifecycle
- **Inference handlers** (`model_fn`, `input_fn`, `predict_fn`, `output_fn`) define how the serving container loads and uses your model
- **Real-time inference** serves low-latency requests through an active endpoint; **Batch Transform** processes S3 datasets offline without one